In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

SEED = 42
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SIZE = 7
WIN = 5
BATCH = 128           # число параллельных партий
ROUNDS = 300          # 300 × 128 = 38 400 партий
LR = 3e-4
ENTROPY_COEF = 0.01

print("Устройство:", DEVICE)

Устройство: cpu


In [ ]:
def make_win_lines(size, win):
    if not 2 <= win <= size:
        raise ValueError("WIN должно быть от 2 до SIZE")

    lines = []
    for row in range(size):
        for col in range(size):
            for dr, dc in [(0, 1), (1, 0), (1, 1), (1, -1)]:
                end_row = row + (win - 1) * dr
                end_col = col + (win - 1) * dc

                if 0 <= end_row < size and 0 <= end_col < size:
                    lines.append([
                        (row + k * dr) * size + col + k * dc
                        for k in range(win)
                    ])
    return torch.tensor(lines, device=DEVICE)


WIN_LINES = make_win_lines(SIZE, WIN)

def has_won(boards, player):
    # boards: [число партий, 49]
    cells = boards[:, WIN_LINES]
    return (cells == player).all(dim=2).any(dim=1)

def winning_cells(boards, player):
    # boards: [число досок, 49]
    lines = boards[:, WIN_LINES]
    almost_won = (
        ((lines == player).sum(dim=2) == WIN - 1)
        & ((lines == 0).sum(dim=2) == 1)
    )

    empty_in_line = almost_won.unsqueeze(2) & (lines == 0)
    indices = WIN_LINES.reshape(1, -1).expand(boards.size(0), -1)

    counts = torch.zeros_like(boards, dtype=torch.long)
    counts.scatter_add_(1, indices, empty_in_line.flatten(1).long())
    return counts > 0

In [ ]:
class PolicyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 1, kernel_size=1),
            nn.Flatten()
        )

    def forward(self, boards, player):
        own = boards == player
        opponent = boards == -player
        x = torch.stack([own, opponent], dim=1).float()
        x = x.reshape(-1, 2, SIZE, SIZE)

        logits = self.net(x)
        return logits.masked_fill(boards != 0, float("-inf"))


agents = {p: PolicyCNN().to(DEVICE) for p in (1, -1)}
optimizers = {
    p: torch.optim.Adam(agents[p].parameters(), lr=LR)
    for p in (1, -1)
}

In [ ]:
DEFENSE_PENALTY = 0.2


def winning_cells(boards, player):
    """Для каждой доски находит клетки, где player может выиграть одним ходом."""
    lines = boards[:, WIN_LINES]

    # В выигрышном отрезке уже стоят WIN - 1 своих фигур и есть одна пустая клетка.
    almost_won = (
        ((lines == player).sum(dim=2) == WIN - 1)
        & ((lines == 0).sum(dim=2) == 1)
    )

    empty_cells = almost_won.unsqueeze(2) & (lines == 0)
    indices = WIN_LINES.reshape(1, -1).expand(boards.size(0), -1)

    # Одна клетка может завершать сразу две линии. Учитываем её один раз.
    counts = torch.zeros_like(boards, dtype=torch.long)
    counts.scatter_add_(1, indices, empty_cells.flatten(1).long())
    return counts > 0


def train_round():
    boards = torch.zeros(
        BATCH, SIZE * SIZE, dtype=torch.int8, device=DEVICE
    )
    active = torch.arange(BATCH, device=DEVICE)
    winners = torch.zeros(BATCH, dtype=torch.int8, device=DEVICE)
    history = {1: [], -1: []}

    # Для каждого агента считаем успешные блокировки и возможности блокировать.
    defense = {
        p: torch.zeros(2, dtype=torch.long, device=DEVICE)
        for p in (1, -1)
    }

    for turn in range(SIZE * SIZE):
        if active.numel() == 0:
            break

        player = 1 if turn % 2 == 0 else -1
        positions = boards[active]

        distribution = Categorical(logits=agents[player](positions, player))
        moves = distribution.sample()

        # Оцениваем выбранный ход до изменения доски.
        with torch.no_grad():
            own_wins = winning_cells(positions, player)
            opponent_wins = winning_cells(positions, -player)
            chosen = moves.unsqueeze(1)

            can_win_now = own_wins.any(dim=1)
            wins_now = own_wins.gather(1, chosen).squeeze(1)
            blocks_threat = opponent_wins.gather(1, chosen).squeeze(1)
            threat_count = opponent_wins.sum(dim=1)

            # Нужно блокировать, если у соперника одна выигрышная клетка,
            # а у нас нет собственной немедленной победы.
            must_block = (threat_count == 1) & ~can_win_now
            defense[player][0] += (must_block & blocks_threat).sum()
            defense[player][1] += must_block.sum()

            # Штрафуем только за угрозу, которую можно было устранить
            # блокировкой или собственной немедленной победой.
            could_avoid_loss = (threat_count == 1) | can_win_now
            threats_left = threat_count - blocks_threat.long()
            missed_defense = could_avoid_loss & ~wins_now & (threats_left > 0)

        history[player].append((
            active,
            distribution.log_prob(moves),
            distribution.entropy(),
            missed_defense,
        ))

        boards[active, moves] = player
        won = has_won(boards[active], player)
        winners[active[won]] = player
        active = active[~won]

    # У каждой сети свои ходы, награды и оптимизатор.
    for player in (1, -1):
        loss = torch.zeros((), device=DEVICE)

        for game_ids, log_probs, entropy, missed_defense in history[player]:
            game_result = winners[game_ids].float() * player
            rewards = game_result - DEFENSE_PENALTY * missed_defense.float()
            loss -= (log_probs * rewards + ENTROPY_COEF * entropy).sum()

        loss = loss / BATCH
        optimizers[player].zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(agents[player].parameters(), 1.0)
        optimizers[player].step()

    return torch.stack([
        (winners == 1).sum(),
        (winners == -1).sum(),
        (winners == 0).sum(),
        defense[1][0], defense[1][1],
        defense[-1][0], defense[-1][1],
    ]).tolist()


def format_defense(blocks, opportunities):
    if opportunities == 0:
        return "нет ситуаций"
    return f"{blocks / opportunities:.1%} ({blocks}/{opportunities})"


# Если предыдущий цикл остановлен после строки «Раунд 20/300»,
# значение 20 продолжит нумерацию с 21. Для нового обучения поставь 0.
START_ROUND = 20

totals = [0] * 7

for epoch in range(START_ROUND + 1, ROUNDS + 1):
    counts = train_round()
    totals = [a + b for a, b in zip(totals, counts)]

    if epoch % 10 == 0 or epoch == ROUNDS:
        x_wins, o_wins, draws, x_blocks, x_chances, o_blocks, o_chances = totals
        games = x_wins + o_wins + draws

        print(
            f"Раунд {epoch:3d}/{ROUNDS} | "
            f"X: {x_wins/games:.1%} | O: {o_wins/games:.1%} | "
            f"Ничьи: {draws/games:.1%}\n"
            f"  Блокировки X: {format_defense(x_blocks, x_chances)} | "
            f"Блокировки O: {format_defense(o_blocks, o_chances)}"
        )

        torch.save({
            "round": epoch,
            "size": SIZE,
            "win": WIN,
            "X": agents[1].state_dict(),
            "O": agents[-1].state_dict(),
            "optimizer_X": optimizers[1].state_dict(),
            "optimizer_O": optimizers[-1].state_dict(),
        }, "tic_tac_toe_7x7.pt")

        totals = [0] * 7

Раунд  30/300 | X: 40.9% | O: 52.8% | Ничьи: 6.2%
  Блокировки X: 8.8% (146/1659) | Блокировки O: 10.5% (170/1617)
Раунд  40/300 | X: 45.7% | O: 49.5% | Ничьи: 4.8%
  Блокировки X: 9.5% (127/1342) | Блокировки O: 12.1% (150/1242)
Раунд  50/300 | X: 50.9% | O: 43.5% | Ничьи: 5.6%
  Блокировки X: 7.5% (87/1164) | Блокировки O: 7.9% (95/1202)
Раунд  60/300 | X: 54.0% | O: 40.5% | Ничьи: 5.5%
  Блокировки X: 8.9% (74/835) | Блокировки O: 8.1% (93/1144)
Раунд  70/300 | X: 52.0% | O: 44.1% | Ничьи: 3.9%
  Блокировки X: 4.9% (33/671) | Блокировки O: 6.3% (71/1119)
Раунд  80/300 | X: 53.9% | O: 42.0% | Ничьи: 4.1%
  Блокировки X: 6.2% (38/611) | Блокировки O: 6.5% (62/958)
Раунд  90/300 | X: 54.4% | O: 44.0% | Ничьи: 1.6%
  Блокировки X: 3.8% (20/529) | Блокировки O: 4.1% (35/853)
Раунд 100/300 | X: 55.5% | O: 43.0% | Ничьи: 1.5%
  Блокировки X: 3.2% (15/474) | Блокировки O: 3.7% (32/866)
Раунд 110/300 | X: 57.5% | O: 41.6% | Ничьи: 0.9%
  Блокировки X: 2.4% (9/379) | Блокировки O: 3.5% (25/71

In [ ]:
@torch.inference_mode()
def evaluate(player, games=500, seed=123):
    rng = torch.Generator(device=DEVICE).manual_seed(seed)
    boards = torch.zeros(games, SIZE * SIZE, dtype=torch.int8, device=DEVICE)
    active = torch.arange(games, device=DEVICE)
    winners = torch.zeros(games, dtype=torch.int8, device=DEVICE)

    for turn in range(SIZE * SIZE):
        if active.numel() == 0:
            break

        current = 1 if turn % 2 == 0 else -1
        positions = boards[active]

        if current == player:
            moves = agents[player](positions, player).argmax(dim=1)
        else:
            moves = torch.multinomial(
                (positions == 0).float(), 1, generator=rng
            ).squeeze(1)

        boards[active, moves] = current
        won = has_won(boards[active], current)
        winners[active[won]] = current
        active = active[~won]

    return {
        "побед": (winners == player).sum().item(),
        "ничьих": (winners == 0).sum().item(),
        "поражений": (winners == -player).sum().item(),
    }


print("Крестики против случайных ноликов:", evaluate(1))
print("Нолики против случайных крестиков:", evaluate(-1))

Крестики против случайных ноликов: {'побед': 491, 'ничьих': 0, 'поражений': 9}
Нолики против случайных крестиков: {'побед': 475, 'ничьих': 0, 'поражений': 25}


In [ ]:
def show_board(board):
    symbols = {0: "·", 1: "X", -1: "O"}
    rows = board.reshape(SIZE, SIZE).cpu().tolist()

    print("\n    " + " ".join(str(i) for i in range(1, SIZE + 1)))
    for i, row in enumerate(rows, start=1):
        print(f"{i:>2}  " + " ".join(symbols[cell] for cell in row))
    print()


@torch.inference_mode()
def play_against_agent():
    while True:
        side = input("За кого играешь? X — крестики, O — нолики: ").strip().upper()
        if side in ("X", "Х"):
            human = 1
            break
        if side in ("O", "О", "0"):
            human = -1
            break
        print("Введи X или O.")

    bot = -human
    board = torch.zeros(1, SIZE * SIZE, dtype=torch.int8, device=DEVICE)
    print(f"\nПоле {SIZE}×{SIZE}. Для победы нужно собрать {WIN} подряд.")
    print("Чтобы закончить игру досрочно, введи «выход».")
    show_board(board)

    for turn in range(SIZE * SIZE):
        player = 1 if turn % 2 == 0 else -1

        if player == human:
            while True:
                answer = input("Твой ход (строка столбец): ").strip()
                if answer.lower() == "выход":
                    print("Игра завершена.")
                    return

                parts = answer.split()
                try:
                    row, col = map(int, parts)
                except ValueError:
                    print("Введи два целых числа через пробел, например 4 4.")
                    continue

                if not (1 <= row <= SIZE and 1 <= col <= SIZE):
                    print(f"Строка и столбец должны быть от 1 до {SIZE}.")
                    continue

                move = (row - 1) * SIZE + (col - 1)
                if board[0, move].item() != 0:
                    print("Клетка занята. Выбери другую.")
                    continue
                break
        else:
            scores = agents[bot](board, bot)
            move = scores.argmax(dim=1).item()
            row, col = divmod(move, SIZE)
            print(f"Ход бота: строка {row + 1}, столбец {col + 1}")

        board[0, move] = player
        show_board(board)

        if has_won(board, player).item():
            print("Ты победил!" if player == human else "Бот победил.")
            return

    print("Ничья!")

За кого играешь? X — крестики, O — нолики: X

Поле 7×7. Для победы нужно собрать 5 подряд.
Чтобы закончить игру досрочно, введи «выход».

    1 2 3 4 5 6 7
 1  · · · · · · ·
 2  · · · · · · ·
 3  · · · · · · ·
 4  · · · · · · ·
 5  · · · · · · ·
 6  · · · · · · ·
 7  · · · · · · ·

Твой ход (строка столбец): 3 4

    1 2 3 4 5 6 7
 1  · · · · · · ·
 2  · · · · · · ·
 3  · · · X · · ·
 4  · · · · · · ·
 5  · · · · · · ·
 6  · · · · · · ·
 7  · · · · · · ·

Ход бота: строка 1, столбец 1

    1 2 3 4 5 6 7
 1  O · · · · · ·
 2  · · · · · · ·
 3  · · · X · · ·
 4  · · · · · · ·
 5  · · · · · · ·
 6  · · · · · · ·
 7  · · · · · · ·

Твой ход (строка столбец): 3 5

    1 2 3 4 5 6 7
 1  O · · · · · ·
 2  · · · · · · ·
 3  · · · X X · ·
 4  · · · · · · ·
 5  · · · · · · ·
 6  · · · · · · ·
 7  · · · · · · ·

Ход бота: строка 1, столбец 2

    1 2 3 4 5 6 7
 1  O O · · · · ·
 2  · · · · · · ·
 3  · · · X X · ·
 4  · · · · · · ·
 5  · · · · · · ·
 6  · · · · · · ·
 7  · · · · · · ·

Твой ход (с

In [ ]:
from shutil import copy2

copy2("tic_tac_toe_7x7.pt", "tic_tac_toe_7x7_round300.pt")

'tic_tac_toe_7x7_round300.pt'

In [ ]:
from pathlib import Path
from shutil import copy2

source = Path("tic_tac_toe_7x7.pt")
backup = Path("tic_tac_toe_7x7_round300.pt")

checkpoint = torch.load(source, map_location=DEVICE, weights_only=True)

if checkpoint["round"] != 300:
    raise ValueError(f"В файле сохранён раунд {checkpoint['round']}, ожидался 300.")

if (checkpoint["size"], checkpoint["win"]) != (SIZE, WIN):
    raise ValueError("SIZE или WIN отличается от настроек сохранённой модели.")

if not backup.exists():
    copy2(source, backup)

for player, label in [(1, "X"), (-1, "O")]:
    agents[player].load_state_dict(checkpoint[label])
    optimizers[player].load_state_dict(checkpoint[f"optimizer_{label}"])

print(f"Оба агента загружены. Раунд 300, поле {SIZE}×{SIZE}, победа — {WIN} подряд.")

Оба агента загружены. Раунд 300, поле 7×7, победа — 5 подряд.


In [ ]:
END_ROUND = 1200
DEFENSE_PENALTY = 0.2
TACTIC_WEIGHT = 1.0

def train_tactical_round():
    boards = torch.zeros(BATCH, SIZE * SIZE, dtype=torch.int8, device=DEVICE)
    active = torch.arange(BATCH, device=DEVICE)
    winners = torch.zeros(BATCH, dtype=torch.int8, device=DEVICE)
    history = {1: [], -1: []}

    # Для каждой стороны: обязательные защиты, выбранные блоки, блоки через argmax.
    stats = torch.zeros(2, 3, dtype=torch.long, device=DEVICE)

    for turn in range(SIZE * SIZE):
        if active.numel() == 0:
            break

        side = turn % 2
        player = 1 if side == 0 else -1
        positions = boards[active]

        logits = agents[player](positions, player)
        distribution = Categorical(logits=logits)
        moves = distribution.sample()

        with torch.no_grad():
            own_wins = winning_cells(positions, player)
            enemy_wins = winning_cells(positions, -player)

            can_win = own_wins.any(dim=1)
            threats = enemy_wins.sum(dim=1)
            must_block = (threats == 1) & ~can_win

            chosen = moves.unsqueeze(1)
            best = logits.argmax(dim=1, keepdim=True)
            wins_now = own_wins.gather(1, chosen).squeeze(1)
            blocks_now = enemy_wins.gather(1, chosen).squeeze(1)
            best_blocks = enemy_wins.gather(1, best).squeeze(1)

            stats[side, 0] += must_block.sum()
            stats[side, 1] += (must_block & blocks_now).sum()
            stats[side, 2] += (must_block & best_blocks).sum()

            # Сохраняем прежний штраф за угрозу, которую можно было устранить.
            avoidable = (threats == 1) | can_win
            missed = avoidable & ~wins_now & (threats > blocks_now.long())

            # Если можно победить, подходит любой выигрышный ход.
            # Иначе учим блокировать единственную непосредственную угрозу.
            tactical_positions = can_win | must_block
            targets = torch.where(can_win[:, None], own_wins, enemy_wins)

        # Здесь градиенты включены: этот сигнал обучает сеть.
        correct_prob = (distribution.probs * targets).sum(dim=1)
        tactical_loss = -correct_prob.clamp_min(1e-8).log()
        tactical_sum = (tactical_loss * tactical_positions).sum()

        history[player].append((
            active, distribution.log_prob(moves), distribution.entropy(),
            missed, tactical_sum, tactical_positions.sum()
        ))

        boards[active, moves] = player
        won = has_won(boards[active], player)
        winners[active[won]] = player
        active = active[~won]

    for player in (1, -1):
        policy_loss = torch.zeros((), device=DEVICE)
        tactic_loss = torch.zeros((), device=DEVICE)
        tactic_count = torch.zeros((), device=DEVICE)

        for ids, log_probs, entropy, missed, tactic_sum, count in history[player]:
            reward = winners[ids].float() * player - DEFENSE_PENALTY * missed.float()
            policy_loss -= (log_probs * reward + ENTROPY_COEF * entropy).sum()
            tactic_loss += tactic_sum
            tactic_count += count

        loss = policy_loss / BATCH + TACTIC_WEIGHT * tactic_loss / tactic_count.clamp_min(1)

        optimizers[player].zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(agents[player].parameters(), 1.0)
        optimizers[player].step()

    return stats


totals = torch.zeros(2, 3, dtype=torch.long, device=DEVICE)

for epoch in range(checkpoint["round"] + 1, END_ROUND + 1):
    totals += train_tactical_round()

    if epoch % 10 == 0 or epoch == END_ROUND:
        print(f"\nРаунд {epoch}/{END_ROUND}")

        for label, (chances, sampled, greedy) in zip(("X", "O"), totals.tolist()):
            if chances:
                print(
                    f"  {label} | Выбранный ход: {sampled/chances:.1%} | "
                    f"Лучший ход: {greedy/chances:.1%} | Ситуаций: {chances}"
                )
            else:
                print(f"  {label} | Обязательных блокировок не было")

        torch.save({
            "round": epoch, "size": SIZE, "win": WIN,
            "X": agents[1].state_dict(),
            "O": agents[-1].state_dict(),
            "optimizer_X": optimizers[1].state_dict(),
            "optimizer_O": optimizers[-1].state_dict(),
        }, "tic_tac_toe_7x7_tactical.pt")

        totals.zero_()


Раунд 310/1200
  X | Выбранный ход: 9.3% | Лучший ход: 13.4% | Ситуаций: 410
  O | Выбранный ход: 8.0% | Лучший ход: 10.8% | Ситуаций: 1373

Раунд 320/1200
  X | Выбранный ход: 7.0% | Лучший ход: 8.2% | Ситуаций: 743
  O | Выбранный ход: 11.1% | Лучший ход: 12.7% | Ситуаций: 1241

Раунд 330/1200
  X | Выбранный ход: 7.6% | Лучший ход: 7.1% | Ситуаций: 1252
  O | Выбранный ход: 12.7% | Лучший ход: 14.6% | Ситуаций: 1274

Раунд 340/1200
  X | Выбранный ход: 10.3% | Лучший ход: 13.7% | Ситуаций: 1233
  O | Выбранный ход: 10.0% | Лучший ход: 11.0% | Ситуаций: 1786

Раунд 350/1200
  X | Выбранный ход: 11.4% | Лучший ход: 15.2% | Ситуаций: 1367
  O | Выбранный ход: 9.7% | Лучший ход: 12.9% | Ситуаций: 1921

Раунд 360/1200
  X | Выбранный ход: 10.4% | Лучший ход: 13.9% | Ситуаций: 1274
  O | Выбранный ход: 9.5% | Лучший ход: 14.3% | Ситуаций: 1949

Раунд 370/1200
  X | Выбранный ход: 9.3% | Лучший ход: 12.9% | Ситуаций: 1210
  O | Выбранный ход: 8.8% | Лучший ход: 12.9% | Ситуаций: 1772

Рау

In [ ]:
from pathlib import Path
from shutil import copy2

save_path = Path("tic_tac_toe_7x7_tactical.pt")
checkpoint = torch.load(save_path, map_location=DEVICE, weights_only=True)

start_round = checkpoint["round"]
end_round = 3200

if (checkpoint["size"], checkpoint["win"]) != (SIZE, WIN):
    raise ValueError("Размер поля или условие победы отличается от сохранения.")

# Сохраняем исходную версию, чтобы потом сравнить, стала ли она играть лучше.
backup_path = save_path.with_name(f"tic_tac_toe_7x7_tactical_round{start_round}.pt")
if not backup_path.exists():
    copy2(save_path, backup_path)

for player, label in [(1, "X"), (-1, "O")]:
    agents[player].load_state_dict(checkpoint[label])
    optimizers[player].load_state_dict(checkpoint[f"optimizer_{label}"])
    agents[player].train()

print(f"Загружен раунд {start_round}. Цель — {end_round}.")
totals = torch.zeros(2, 3, dtype=torch.long, device=DEVICE)

for epoch in range(start_round + 1, end_round + 1):
    totals += train_tactical_round()

    if epoch % 10 == 0 or epoch == end_round:
        print(f"\nРаунд {epoch}/{end_round}")

        for label, (chances, sampled, greedy) in zip(("X", "O"), totals.tolist()):
            if chances:
                print(
                    f"  {label} | Выбранный ход: {sampled/chances:.1%} | "
                    f"Лучший ход: {greedy/chances:.1%} | Ситуаций: {chances}"
                )
            else:
                print(f"  {label} | Обязательных блокировок не было")

        # Сначала пишем во временный файл, затем заменяем последнее сохранение.
        temp_path = save_path.with_suffix(".tmp")
        torch.save({
            "round": epoch,
            "size": SIZE,
            "win": WIN,
            "X": agents[1].state_dict(),
            "O": agents[-1].state_dict(),
            "optimizer_X": optimizers[1].state_dict(),
            "optimizer_O": optimizers[-1].state_dict(),
        }, temp_path)
        temp_path.replace(save_path)
        totals.zero_()

Загружен раунд 1200. Цель — 3200.

Раунд 1210/3200
  X | Выбранный ход: 14.3% | Лучший ход: 16.9% | Ситуаций: 692
  O | Выбранный ход: 16.1% | Лучший ход: 20.4% | Ситуаций: 685

Раунд 1220/3200
  X | Выбранный ход: 15.9% | Лучший ход: 19.9% | Ситуаций: 674
  O | Выбранный ход: 17.4% | Лучший ход: 23.2% | Ситуаций: 746

Раунд 1230/3200
  X | Выбранный ход: 17.9% | Лучший ход: 25.0% | Ситуаций: 597
  O | Выбранный ход: 19.4% | Лучший ход: 26.9% | Ситуаций: 670

Раунд 1240/3200
  X | Выбранный ход: 18.1% | Лучший ход: 20.2% | Ситуаций: 609
  O | Выбранный ход: 21.4% | Лучший ход: 29.6% | Ситуаций: 768

Раунд 1250/3200
  X | Выбранный ход: 15.2% | Лучший ход: 18.6% | Ситуаций: 710
  O | Выбранный ход: 22.7% | Лучший ход: 27.5% | Ситуаций: 794

Раунд 1260/3200
  X | Выбранный ход: 16.3% | Лучший ход: 20.3% | Ситуаций: 790
  O | Выбранный ход: 24.3% | Лучший ход: 26.8% | Ситуаций: 717

Раунд 1270/3200
  X | Выбранный ход: 18.2% | Лучший ход: 20.9% | Ситуаций: 740
  O | Выбранный ход: 22.7% |

In [ ]:
play_against_agent()

За кого играешь? X — крестики, O — нолики: X

Поле 7×7. Для победы нужно собрать 5 подряд.
Чтобы закончить игру досрочно, введи «выход».

    1 2 3 4 5 6 7
 1  · · · · · · ·
 2  · · · · · · ·
 3  · · · · · · ·
 4  · · · · · · ·
 5  · · · · · · ·
 6  · · · · · · ·
 7  · · · · · · ·

Твой ход (строка столбец): 4 4

    1 2 3 4 5 6 7
 1  · · · · · · ·
 2  · · · · · · ·
 3  · · · · · · ·
 4  · · · X · · ·
 5  · · · · · · ·
 6  · · · · · · ·
 7  · · · · · · ·

Ход бота: строка 5, столбец 3

    1 2 3 4 5 6 7
 1  · · · · · · ·
 2  · · · · · · ·
 3  · · · · · · ·
 4  · · · X · · ·
 5  · · O · · · ·
 6  · · · · · · ·
 7  · · · · · · ·

Твой ход (строка столбец): 4 5

    1 2 3 4 5 6 7
 1  · · · · · · ·
 2  · · · · · · ·
 3  · · · · · · ·
 4  · · · X X · ·
 5  · · O · · · ·
 6  · · · · · · ·
 7  · · · · · · ·

Ход бота: строка 4, столбец 3

    1 2 3 4 5 6 7
 1  · · · · · · ·
 2  · · · · · · ·
 3  · · · · · · ·
 4  · · O X X · ·
 5  · · O · · · ·
 6  · · · · · · ·
 7  · · · · · · ·

Твой ход (с